# Module 7 · Auto-summary chatbot (context management)

Every N user turns a separate call writes a 2-sentence summary. In *Summary + last K* mode the model receives only that summary plus the latest messages, so the prompt stops growing. This is the Colab version of the app's chatbot tab.

> 🔐 **Keys** — nothing in this notebook contains a key. In Colab open the **🔑 Secrets** panel (left sidebar), click **+ Add new secret**,
> type the *name* exactly as below, paste the *value*, and switch on **Notebook access**. If you skip this, you'll be asked once (hidden input).

| Secret name | What it is | Get it at |
|---|---|---|
| `GEMINI_API_KEY` | Google Gemini API key | https://aistudio.google.com/apikey |

**Never** paste a key into a code cell, commit it to GitHub, or share a notebook that shows it in an output. If one leaks, delete it in the provider's console and create a new one.

In [ ]:
!pip install -q google-genai pandas

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=get_secret("GEMINI_API_KEY", hint="(from aistudio.google.com/apikey) "))
# Change to "gemini-2.5-flash" if your key does not have access to the newer model.
MODEL = "gemini-3.5-flash-lite"
print('Ready. Model =', MODEL)

## 1 · Settings (edit and re-run)

In [ ]:
SUMMARIZE_EVERY = 3     # write a summary every N user turns
KEEP_LAST = 4           # messages kept verbatim in summary mode
MODE = "summary"        # "full" or "summary"

## 2 · The chatbot

In [ ]:
import pandas as pd
msgs, summary, turns, stats = [], "", 0, []

def contents_to_send():
    recent = msgs if (MODE == "full" or not summary) else msgs[-KEEP_LAST:]
    pre = []
    if MODE != "full" and summary:
        pre = [types.Content(role="user", parts=[types.Part(text=f"(Context summary of earlier conversation: {summary})")]),
               types.Content(role="model", parts=[types.Part(text="Understood.")])]
    return pre + [types.Content(role=m["role"], parts=[types.Part(text=m["text"])]) for m in recent]

def chat(user_text):
    global summary, turns
    msgs.append({"role": "user", "text": user_text}); turns += 1
    sent = contents_to_send()
    r = client.models.generate_content(model=MODEL, contents=sent)
    msgs.append({"role": "model", "text": r.text})
    tokens = getattr(r.usage_metadata, "prompt_token_count", None)
    stats.append({"turn": turns, "mode": "summary + recent" if (MODE != "full" and summary) else "full history",
                  "messages sent": len(sent), "tokens sent": tokens})
    if turns % SUMMARIZE_EVERY == 0:
        convo = "\n".join(f"{m['role']}: {m['text']}" for m in msgs)
        prev = f"Previous summary: {summary}\n\n" if summary else ""
        summary = client.models.generate_content(
            model=MODEL, contents=prev + "Summarize the conversation below in AT MOST 2 short sentences (under 45 words). "
                                          "Keep names, numbers and preferences.\n\n" + convo).text.strip()
    return r.text

## 3 · Scripted demo (no typing needed)

In [ ]:
for q in ["Hi, I'm Priya from Pune.", "I love trekking and poha.", "I have a dog called Bruno.", "I work as a data analyst.",
          "I'm planning a trip in December.", "What do you remember about me?"]:
    print("YOU:", q); print("AI :", chat(q).strip()[:300], "\n")
print("SUMMARY:", summary)
pd.DataFrame(stats)

## 4 · Chat yourself
Run the cell, type a message, press Enter. Type `quit` to stop.

In [ ]:
while True:
    t = input("You: ").strip()
    if t.lower() in {"quit", "exit", ""}:
        break
    print("AI :", chat(t).strip(), "\n")
pd.DataFrame(stats)

Compare: set `MODE = "full"`, re-run sections 2–3 and watch **tokens sent** keep climbing, while summary mode levels off.